<a href="https://colab.research.google.com/github/jhughes7386/cosc-650-applied-llm-systems/blob/week-03/week-03/week3_prompt_engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 3 (starter): Prompts as Engineering Artifacts

Runs without an API key: the semantic metric is local, and the model calls fall back to clearly-labeled fixtures so you can see the harness work. Set `GEMINI_API_KEY` to run the prompts for real. Cells marked **TODO (you)** are yours.

Dependencies: `sentence-transformers` (local). For live calls: `pip install openai` and a Gemini key.

In [12]:
import os, json, pathlib
from google.colab import userdata

def gemini_chat(messages, model='gemini-3.5-flash-lite', **kw):
    """Gemini via the OpenAI-compatible endpoint. Returns text, or None if no key (API-BLOCKED)."""
    key = userdata.get('GEMINI_API_KEY')
    if not key:
        return None
    from openai import OpenAI
    client = OpenAI(
        api_key=key,
        base_url='https://generativelanguage.googleapis.com/v1beta/openai/'
    )
    return client.chat.completions.create(
        model=model,
        messages=messages,
        **kw
    ).choices[0].message.content

LIVE = key is not None
print('live model calls:', LIVE, '(fixtures used when False)')

live model calls: True (fixtures used when False)


In [2]:
os.environ['HF_HOME'] = str((pathlib.Path('.') / '.hf_cache').resolve())
from sentence_transformers import SentenceTransformer, util
emb = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
def exact_match(a, b):
    return float(str(a).strip().lower() == str(b).strip().lower())
def semantic_sim(a, b):
    e = emb.encode([a, b], convert_to_tensor=True, normalize_embeddings=True)
    return round(float(util.cos_sim(e[0], e[1])), 3)
print('metrics ready (exact-match + semantic)')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

metrics ready (exact-match + semantic)


## Part 1: Versioned prompts and a test suite
**TODO (you):** in your repo, store each prompt version as its own file. Here they are inline so the notebook runs. Task: classify a support ticket. v2 adds an intent rule.

In [3]:
PROMPT_V1 = '''
System: You classify movie requests into one of these categories: action, comedy, horror, romance, family.
Return JSON with exactly two fields: category and rationale.
Briefly explain your classification in the rationale.

Example 1:
Movie request: I want a movie with lots of explosions and exciting fight scenes.
Answer: {"category": "action", "rationale": "The user wants an action movie."}

Example 2:
Movie request: I want something funny that will make me laugh.
Answer: {"category": "comedy", "rationale": "The user wants a funny movie."}

Example 3:
Movie request: I want a scary movie that will keep me up at night.
Answer: {"category": "horror", "rationale": "The user wants a scary movie."}

Think about the main type of movie the user is asking for, then give a brief rationale.
'''

PROMPT_V2 = '''
System: You classify movie requests into one of these categories: action, comedy, horror, romance, family.
Return JSON with exactly two fields: category and rationale.
Briefly explain your classification in the rationale.

Example 1:
Movie request: I want a movie with lots of explosions and exciting fight scenes.
Answer: {"category": "action", "rationale": "The user wants an action movie."}

Example 2:
Movie request: I want something funny that will make me laugh.
Answer: {"category": "comedy", "rationale": "The user wants a funny movie."}

Example 3:
Movie request: I want a scary movie that will keep me up at night.
Answer: {"category": "horror", "rationale": "The user wants a scary movie."}

When multiple genres are mentioned, classify based on the main type of movie the person wants, not just a genre mentioned in the request.

Think about the main type of movie the user is asking for, then give a brief rationale.
'''

tests = [

  {'id':1,'ticket':'I want a movie with lots of explosions and exciting fight scenes.','cat':'action','why':'action movie'},

  {'id':2,'ticket':'I want something funny that will make me laugh.','cat':'comedy','why':'funny movie'},

  {'id':3,'ticket':'I want a scary movie that will keep me up at night.','cat':'horror','why':'scary movie'},

  {'id':4,'ticket':'I want a romantic movie with a love story.','cat':'romance','why':'romantic love story'},

  {'id':5,'ticket':'I need a movie that the whole family can watch together.','cat':'family','why':'family friendly movie'},

  {'id':6,'ticket':'I want a funny movie with a little bit of romance.','cat':'comedy','why':'funny movie with some romance'},

  {'id':7,'ticket':'I want an action movie with a romantic story between the main characters.','cat':'action','why':'action movie with romance'},

  {'id':8,'ticket':'I want something scary, but I also want some funny moments.','cat':'horror','why':'scary movie with some comedy'},

  {'id':9,'ticket':'I want a family movie that also has some funny characters.','cat':'family','why':'family movie with some comedy'},

  {'id':10,'ticket':'I want a romantic comedy about two people falling in love.','cat':'romance','why':'romantic love story'},

]

print('prompt versions:', 2, '| test cases:', len(tests))

prompt versions: 2 | test cases: 10


In [4]:
# Labeled fixtures stand in for model output when LIVE is False. v2 improves #7 but regresses #8.
FIX = {
  'v1': {
    1:('action','action movie'),
    2:('comedy','funny movie'),
    3:('horror','scary movie'),
    4:('romance','romantic love story'),
    5:('family','family friendly movie'),
    6:('comedy','funny movie with some romance'),
    7:('romance','romantic story with action'),
    8:('horror','scary movie with some comedy'),
    9:('comedy','funny family movie'),
    10:('comedy','funny movie about romance')
  },
  'v2': {
    1:('action','action movie'),
    2:('comedy','funny movie'),
    3:('horror','scary movie'),
    4:('romance','romantic love story'),
    5:('family','family friendly movie'),
    6:('comedy','funny movie with some romance'),
    7:('action','action movie with romance'),
    8:('comedy','funny movie with some horror'),
    9:('family','family movie with some comedy'),
    10:('romance','romantic love story')
  },
}

def run_case(version, prompt, t):
    if LIVE:
        txt = gemini_chat([{'role':'user','content': prompt + '\nTicket: ' + t['ticket']}])
        try:
            d = json.loads(txt); return d.get('category',''), d.get('rationale','')
        except Exception:
            return '', txt or ''
    return FIX[version][t['id']]

def score(version, prompt):
    rows = []
    for t in tests:
        cat, why = run_case(version, prompt, t)
        rows.append({'id':t['id'],'exact':exact_match(t['cat'],cat),'sem':semantic_sim(t['why'],why),'got':cat})
    acc = sum(r['exact'] for r in rows)/len(rows)
    return acc, rows

acc1, r1 = score('v1', PROMPT_V1)
acc2, r2 = score('v2', PROMPT_V2)
print(f'v1 exact-match {acc1:.0%}   v2 exact-match {acc2:.0%}')

v1 exact-match 70%   v2 exact-match 90%


### Prompt Versioning

I saved each prompt version as a separate file so the changes between versions can be tracked. For this experiment, I used a simple movie genre classification task. The prompt takes a movie request, such as wanting a funny or scary movie, and classifies it into one of five genres. Version 2 adds a rule for handling requests that mention multiple genres.


## Part 2: Build the test suite
Write at least ten input and expected-output pairs. Score each response with two metrics: one exact-match check on the structured field, and one semantic-similarity score (sentence-transformers, run locally on CPU).



In [5]:
tests = [

  {'id':1,'ticket':'I want a movie with lots of explosions and exciting fight scenes.','cat':'action','why':'action movie'},

  {'id':2,'ticket':'I want something funny that will make me laugh.','cat':'comedy','why':'funny movie'},

  {'id':3,'ticket':'I want a scary movie that will keep me up at night.','cat':'horror','why':'scary movie'},

  {'id':4,'ticket':'I want a romantic movie with a love story.','cat':'romance','why':'romantic love story'},

  {'id':5,'ticket':'I need a movie that the whole family can watch together.','cat':'family','why':'family friendly movie'},

  {'id':6,'ticket':'I want a funny movie with a little bit of romance.','cat':'comedy','why':'funny movie with some romance'},

  {'id':7,'ticket':'I want an action movie with a romantic story between the main characters.','cat':'action','why':'action movie with romance'},

  {'id':8,'ticket':'I want something scary, but I also want some funny moments.','cat':'horror','why':'scary movie with some comedy'},

  {'id':9,'ticket':'I want a family movie that also has some funny characters.','cat':'family','why':'family movie with some comedy'},

  {'id':10,'ticket':'I want a romantic comedy about two people falling in love.','cat':'romance','why':'romantic love story'}
]

print('test cases:', len(tests))

test cases: 10


In [14]:
acc1, r1 = score('v1', PROMPT_V1)
acc2, r2 = score('v2', PROMPT_V2)

print(f'v1 exact-match: {acc1:.0%}')
print(f'v2 exact-match: {acc2:.0%}')

print('\nv1 semantic similarity:')
print([r['sem'] for r in r1])

print('\nv2 semantic similarity:')
print([r['sem'] for r in r2])

v1 exact-match: 80%
v2 exact-match: 80%

v1 semantic similarity:
[0.619, 0.345, 0.602, 0.687, 0.454, 0.276, 0.611, 0.515, 0.547, 0.412]

v2 semantic similarity:
[0.622, 0.539, 0.6, 0.647, 0.667, 0.404, 0.61, 0.331, 0.58, 0.663]


### Test Results

The test suite contains 10 cases. Version 1 and Version 2 both had an exact-match score of 80%. The semantic similarity scores were also calculated for each case. Version 2 had higher semantic similarity on several test cases, but the exact-match score stayed the same between the two versions. I will use the individual test results to look at which cases improved or regressed after the prompt change.


## Part 3 and 4: the tradeoff and the failure
Show one case the edit improved and one it regressed. The regression is your required failure.

In [15]:
for t in tests:
    e1 = next(r for r in r1 if r['id']==t['id'])['exact']
    e2 = next(r for r in r2 if r['id']==t['id'])['exact']
    if e1 != e2:
        verdict = 'IMPROVED' if e2 > e1 else 'REGRESSED'
        print(f"#{t['id']} expected {t['cat']!r}: v1 {'ok' if e1 else 'miss'} -> v2 {'ok' if e2 else 'miss'}  [{verdict}]")
# TODO (you): explain why v2 helped one case and hurt another, and how you would resolve the tradeoff.

#5 expected 'family': v1 miss -> v2 ok  [IMPROVED]
#8 expected 'horror': v1 ok -> v2 miss  [REGRESSED]


### Failure Case and Tradeoff

The regression seen in Part 3 is from test case 8. In Version 1, the model correctly classified the request as horror because the main request was for something scary, even though the user also wanted some funny moments. In Version 2, I added a rule telling the model to focus on the main type of movie when multiple genres are mentioned. This helped test case 5 because the model correctly classified the family movie instead of focusing on the funny characters. However, this same change hurt test case 8 because the request included both horror and comedy, and the model determined that comedy was the main genre instead of the horror category I expected.

The results show that adding a more specific rule can improve some ambiguous cases but also create problems in other instances. Both versions had an exact-match score of 80%, so the overall score did not improve even though one case improved and another regressed. Instead of continuing to add more rules every time a failure occurs, I would use example-based prompting to show the model how I want these ambiguous cases handled. Continuously adding rules could make the prompt too long or introduce contradictory instructions that make the model less consistent. For example, I could add another case showing a request with multiple genres where one genre is clearly the main request. I would then add more test cases with multiple genres and retest the evaluation to see if the example improves the tradeoff without creating unforeseen errors.


## Part 5: Submit
Store the prompt versions as files, run the suite (set your key for real calls), and open a pull request with the metric numbers and a linked research note. Rubric: versioned prompts (15), structured prompt (20), test suite with two metrics (25), tradeoff with numbers (25), PR hygiene (15).